In [30]:
import pandas as pd 
import numpy as np 
from collections import deque
import os

### Starting the simulation 

In [27]:
def fixed_simulation(dataset):
    ## Using the same metrics 
    records = []
    waiting_times = [] ## Calculating the total time student had to wait 
    lunch_duration = 120 ## The lunch will run for 120 minutes 
    cooking_time = 10 ## Total time required to cook the food 
    batch_size = 65 ## Portions produced per batch 
    available_portions = 65 ## Food currently ready to serve 
    cooking_finish_time = None ## No batch is getting cooked currently 
    scheduled_interval = 15 ## Starting cooking batch every 15 minute
    queue = deque() ## Starting unique queue for each of the input
    batches_started = 0

    for _, row in dataset.sort_values("minutes").iterrows():
        minute = int(row['minutes']) ## Minute count 
        arrivals = int(row['arriving_students']) ## Total number of students arriving in that minute 
        ### Adding the food 
        if cooking_finish_time is not None: ## If something is getting cooked
            if minute >= cooking_finish_time: ## If the minute or time
                available_portions += batch_size ## We add the total food to the batch size 
                cooking_finish_time = None 
        queue.extend([minute] * arrivals) ## Extend queue by each people arrivals per minute. if 3 people come in minute 10, the queue will be extended by [10,10,10]
        food_served = min(available_portions, len(queue)) ## The food will be served if there is available 
        for _ in range(food_served): ## Looping through the ones who got served
            arrival_minute = queue.popleft() ## Finding the minute they came in
            waiting_times.append(minute-arrival_minute) ## Minute now and the time they were served and appending the waiting time to the list
        available_portions -= food_served ## Decreasing the available_portions with the food served

        ### When to cook the batch 
        if (cooking_finish_time is None and minute % scheduled_interval == 0 and minute + cooking_time <= lunch_duration ):
            cooking_finish_time = minute + cooking_time ## If the minute is 30 and the food is need at minute 30 it will take 10 minutes to make the food meaning the fodd will be available at 30 + 10 = 40 
            batches_started += 1 ## Total number of batches cooked 


        records.append({
                        "minute": minute,
                        "arrivals": arrivals,
                        "fixed_cooking_minute": 15,
                        "available_portions": available_portions,
                        "students_waiting": len(queue),
                        "cooking": cooking_finish_time is not None
        })
    simulation_result = pd.DataFrame(records)
    return simulation_result, waiting_times

### Running model through each data 

In [ ]:
## storing the data in the list 
dataset_path = '../CafeteriaSimulation/simulation_datas/'
datasets = []

for filename in os.listdir(dataset_path):
    if filename.endswith('.csv'):
        filepath = os.path.join(dataset_path, filename) ### Creating a file path for each directory
        data = pd.read_csv(filepath)
        datasets.append(data)

